In [7]:
import pyabc
pyabc.__version__

['clone_competition_simulation.fitness',
 'clone_competition_simulation.fitness.fitness_classes',
 'clone_competition_simulation.parameters.fitness_validation',
 'clone_competition_simulation.simulation_algorithms.general_2D_class',
 'clone_competition_simulation.simulation_algorithms.general_differentiated_cell_class',
 'clone_competition_simulation.simulation_algorithms.general_sim_class']

In [10]:
# ✅ Correct imports for the API shown by inspect.signature(Parameters)

from clone_competition_simulation.parameters import Parameters
from clone_competition_simulation.parameters.times_validation import TimeParameters
from clone_competition_simulation.parameters.population_validation import PopulationParameters
from clone_competition_simulation.parameters.fitness_validation import FitnessParameters

# Now locate Gene / MutationGenerator / NormalDist in YOUR install:
import pkgutil, clone_competition_simulation as ccs

mods = sorted([m.name for m in pkgutil.walk_packages(ccs.__path__, ccs.__name__ + ".")])
# show only likely modules
[x for x in mods if any(k in x.lower() for k in ["fitness", "mutation", "dist", "gene"])]

['clone_competition_simulation.fitness',
 'clone_competition_simulation.fitness.fitness_classes',
 'clone_competition_simulation.parameters.fitness_validation',
 'clone_competition_simulation.simulation_algorithms.general_2D_class',
 'clone_competition_simulation.simulation_algorithms.general_differentiated_cell_class',
 'clone_competition_simulation.simulation_algorithms.general_sim_class']

In [ ]:
import os
import numpy as np
import pandas as pd

from pyabc import ABCSMC, Distribution, RV
from pyabc.sampler import MulticoreEvalParallelSampler

from clone_competition_simulation.parameters import Parameters
from clone_competition_simulation.parameters.times_validation import TimeParameters
from clone_competition_simulation.parameters.population_validation import PopulationParameters
from clone_competition_simulation.parameters.fitness_validation import FitnessParameters
from clone_competition_simulation.fitness.fitness_classes import Gene, MutationGenerator, NormalDist

# ----------------------------
# CONFIG
# ----------------------------
CSV = "dnds_long_removed_outliers.csv"
GENE = "AJUBA"                 # change to NOTCH1 / TP53 etc.
OUTDIR = f"abc_{GENE}"
os.makedirs(OUTDIR, exist_ok=True)

AGE_BINS   = [(20,29),(30,39),(40,49),(50,59),(60,120)]
AGE_LABELS = np.array([25,35,45,55,70], dtype=float)

GRID_SIDE = 50
MAX_AGE = 80
DIVISION_RATE = 50

FIT_VAR = 0.02
SYN_PROP = 0.4
MIN_SIZE_DNDS = 10   # A1 stabilised

# ABC controls
POP_SIZE = 200
N_PROCS = 8
MIN_EPS = 0.15
MAX_POPS = 10

# ----------------------------
# Data -> observed summary (log10 median dN/dS per bin)
# ----------------------------
df = pd.read_csv(CSV)
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["gene_name","donor_id","Age","dnds"])
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["dnds"] = pd.to_numeric(df["dnds"], errors="coerce")
df = df.dropna(subset=["Age","dnds"])

df_gene = df[df["gene_name"] == GENE].copy()

def make_target_curve(dfg):
    d = dfg.copy()
    d = d[(d["dnds"] > 0) & np.isfinite(d["dnds"]) & np.isfinite(d["Age"])]
    y = []
    for lo, hi in AGE_BINS:
        vals = d.loc[(d["Age"]>=lo) & (d["Age"]<=hi), "dnds"].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals) & (vals > 0)]
        y.append(np.nan if vals.size == 0 else np.log10(np.median(vals)))
    return np.array(y, dtype=float)

y_obs = make_target_curve(df_gene)
print("Observed:", y_obs, "finite bins:", np.isfinite(y_obs).sum())

y_obs_dict = {f"y{i}": (float(y_obs[i]) if np.isfinite(y_obs[i]) else np.nan)
              for i in range(len(AGE_LABELS))}

# ----------------------------
# Simulator -> summary stats
# ----------------------------
def sim_gene_curve(gene, seed, mu, fit_mean, ages=AGE_LABELS, min_size=MIN_SIZE_DNDS):
    np.random.seed(seed)

    mut_gen = MutationGenerator(
        genes=[
            Gene(
                name=gene,
                mutation_distribution=NormalDist(mean=float(fit_mean), var=float(FIT_VAR)),
                synonymous_proportion=float(SYN_PROP),
                weight=1
            )
        ]
    )

    p = Parameters(
        algorithm="WF2D",
        times=TimeParameters(max_time=MAX_AGE, division_rate=DIVISION_RATE),
        population=PopulationParameters(
            initial_cells=GRID_SIDE * GRID_SIDE,
            cell_in_own_neighbourhood=True
        ),
        fitness=FitnessParameters(
            mutation_rates=float(mu),
            mutation_generator=mut_gen
        )
    )

    sim = p.get_simulator()
    sim.run_sim()

    y = []
    for a in ages:
        dnds = sim.get_dnds(t=float(a), gene=gene, min_size=min_size)
        if (dnds is None) or (not np.isfinite(dnds)) or (dnds <= 0):
            y.append(np.nan)
        else:
            y.append(np.log10(dnds))
    return np.array(y, dtype=float)

def rmse_nan_safe(sim_y, obs_y, min_bins=3):
    m = np.isfinite(sim_y) & np.isfinite(obs_y)
    if m.sum() < min_bins:
        return 1e6
    return float(np.sqrt(np.mean((sim_y[m] - obs_y[m])**2)))

def model(pars):
    mu = float(pars["mu"])
    fit_mean = float(pars["fit_mean"])
    seed = np.random.randint(0, 2**31 - 1)

    y_sim = sim_gene_curve(GENE, seed=seed, mu=mu, fit_mean=fit_mean)

    return {f"y{i}": (float(y_sim[i]) if np.isfinite(y_sim[i]) else np.nan)
            for i in range(len(AGE_LABELS))}

def distance(sim, obs):
    sim_y = np.array([sim[f"y{i}"] for i in range(len(AGE_LABELS))], dtype=float)
    obs_y = np.array([obs[f"y{i}"] for i in range(len(AGE_LABELS))], dtype=float)
    return rmse_nan_safe(sim_y, obs_y, min_bins=3)

# ----------------------------
# Priors
# ----------------------------
priors = Distribution(
    mu=RV("loguniform", 1e-6, 1e-2),
    fit_mean=RV("uniform", 1.0, 0.30)
)

# ----------------------------
# Run ABC-SMC
# ----------------------------
sampler = MulticoreEvalParallelSampler(n_procs=N_PROCS)
abc = ABCSMC(model, priors, distance, population_size=POP_SIZE, sampler=sampler)

db_path = "sqlite:///" + os.path.join(OUTDIR, f"{GENE}_abc.db")
abc.new(db_path, y_obs_dict)

history = abc.run(minimum_epsilon=MIN_EPS, max_nr_populations=MAX_POPS)

print("✅ Done. DB:", db_path)

# ----------------------------
# Save posterior particles
# ----------------------------
df_post, w = history.get_distribution()
df_post["weight"] = w
post_path = os.path.join(OUTDIR, f"{GENE}_posterior.csv")
df_post.to_csv(post_path, index=False)
print("✅ Posterior saved:", post_path)

# Quick weighted quantiles
def weighted_quantile(x, w, qs):
    x = np.asarray(x); w = np.asarray(w)
    idx = np.argsort(x)
    x = x[idx]; w = w[idx]
    cdf = np.cumsum(w) / np.sum(w)
    return np.interp(qs, cdf, x)

for p in ["mu", "fit_mean"]:
    q = weighted_quantile(df_post[p].values, w, [0.1, 0.5, 0.9])
    print(p, "10/50/90:", q)

ABC.Sampler INFO: Parallelize sampling on 8 processes.
ABC.History INFO: Start <ABCSMC id=2, start_time=2026-03-03 02:09:09>
ABC INFO: Calibration sample t = -1.


Observed: [2.88165818 2.8550826  2.73008546 3.18739026 3.02755149] finite bins: 5


2026-03-03 02:09:09.170 | DEBUG    | clone_competition_simulation.parameters.population_validation:_try_making_square_grid:89 - Using a grid of 50x50
2026-03-03 02:09:09.174 | DEBUG    | clone_competition_simulation.parameters.population_validation:_try_making_square_grid:89 - Using a grid of 50x50
2026-03-03 02:09:09.181 | DEBUG    | clone_competition_simulation.parameters.population_validation:_try_making_square_grid:89 - Using a grid of 50x50
2026-03-03 02:09:09.190 | DEBUG    | clone_competition_simulation.parameters.population_validation:_try_making_square_grid:89 - Using a grid of 50x50
2026-03-03 02:09:09.198 | DEBUG    | clone_competition_simulation.parameters.population_validation:_try_making_square_grid:89 - Using a grid of 50x50
2026-03-03 02:09:09.208 | DEBUG    | clone_competition_simulation.parameters.population_validation:_try_making_square_grid:89 - Using a grid of 50x50
2026-03-03 02:09:09.219 | DEBUG    | clone_competition_simulation.parameters.population_validation:_